# Aula 4, Sentence Transformers

Notebook executável que acompanha a aula [04-sentence-transformers.md](../../lessons/modulo-04-word-embeddings/04-sentence-transformers.md).

## O que vamos fazer aqui

Agora subimos da palavra para a frase. Vamos construir uma busca semântica sobre
perguntas de alunos: dada uma consulta, encontrar as perguntas mais parecidas pelo
sentido, e não pelas palavras exatas. Esse é o projeto que fecha o módulo.

Para rodar sem baixar modelos grandes, montamos vetores de palavra a partir das
coocorrências do próprio conjunto e representamos cada pergunta pela média dos seus
vetores. No fim, há um caminho opcional com a biblioteca sentence-transformers, que faz
o mesmo com um modelo de verdade.

## Vetores de palavra a partir do conjunto de perguntas

Reusamos a receita da aula de GloVe, agora sobre o corpus das perguntas: coocorrência
mais SVD, gerando um vetor para cada palavra que aparece nas perguntas.

In [ ]:
import numpy as np
import re

perguntas = [
    "como faço a derivada de uma função",
    "qual a regra da cadeia na derivada",
    "como resolvo um sistema linear com matrizes",
    "o que é um autovetor de uma matriz",
    "como declaro uma função em python",
    "o que é um laço de repetição em python",
]


def tokenizar(texto):
    return re.findall(r"\w+", texto.lower())


tokens = [tokenizar(p) for p in perguntas]
vocab = sorted({w for t in tokens for w in t})
vi = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

CO = np.zeros((V, V))
for t in tokens:
    ids = [vi[w] for w in t]
    for i, c in enumerate(ids):
        for j in range(max(0, i - 2), min(len(ids), i + 3)):
            if j != i:
                CO[c, ids[j]] += 1
U, S, _ = np.linalg.svd(np.log1p(CO))
E = U[:, :8] * S[:8]
print("vetores de palavra prontos:", E.shape)

Cada palavra das perguntas agora tem um vetor de 8 dimensões. O próximo passo
é combinar os vetores das palavras de uma pergunta em um único vetor de frase.

## Vetor de frase e busca semântica

O vetor de uma frase é a média dos vetores das suas palavras. Para buscar, comparamos o
vetor da consulta com o de cada pergunta pela similaridade do cosseno e ordenamos do
mais parecido ao menos parecido.

In [ ]:
def vetor_frase(texto):
    """Vetor da frase como média dos vetores das suas palavras."""
    vetores = [E[vi[w]] for w in tokenizar(texto) if w in vi]
    return np.mean(vetores, axis=0) if vetores else np.zeros(8)


def cosseno(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(a @ b / (na * nb)) if na and nb else 0.0


base = [vetor_frase(p) for p in perguntas]


def buscar(consulta):
    q = vetor_frase(consulta)
    pontuadas = [(cosseno(q, base[i]), perguntas[i]) for i in range(len(perguntas))]
    return sorted(pontuadas, reverse=True)


print("Consulta: 'como calculo a derivada'\n")
for score, pergunta in buscar("como calculo a derivada"):
    print(f"{score:.3f}  {pergunta}")

As duas perguntas sobre derivada encabeçam o ranking, à frente das de álgebra
e programação, mesmo a consulta não repetindo todas as palavras delas. É a busca por
sentido funcionando. Não é perfeita, pois a média de vetores é simples e o conjunto é
pequeno, mas já entrega o comportamento certo.

## Caminho opcional, com sentence-transformers

Esta célula faz a mesma busca com um modelo de verdade, que gera embeddings de frase de
alta qualidade. Ela roda apenas se a biblioteca estiver instalada, e exibe um aviso
caso contrário.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer, util

    modelo = SentenceTransformer('all-MiniLM-L6-v2')
    emb_base = modelo.encode(perguntas, convert_to_tensor=True)
    consulta = 'como calculo a derivada'
    emb_q = modelo.encode(consulta, convert_to_tensor=True)
    scores = util.cos_sim(emb_q, emb_base)[0]
    ordem = scores.argsort(descending=True)
    print(f"Consulta: {consulta!r} (com sentence-transformers)\n")
    for i in ordem:
        print(f"{scores[i]:.3f}  {perguntas[i]}")
except Exception as erro:
    print('sentence-transformers não disponível, usando só a média de vetores acima.')
    print('Para instalar: pip install sentence-transformers. Detalhe:', erro)

Com o modelo de verdade, a busca fica bem mais robusta, inclusive com
consultas que usam sinônimos. Para o projeto do módulo, compare a busca semântica com
uma busca por palavra-chave e mostre um caso em que a semântica acerta e a de
palavra-chave falha, explicando por quê.